# Importing the necessary libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import optuna
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
print(device)

cuda


# Loading the datasets

In [4]:
blr_df = pd.read_csv('../Data/Processed/blr_df_enhanced.csv')
hyd_df = pd.read_csv('../Data/Processed/hyd_df_enhanced.csv')
pune_df = pd.read_csv('../Data/Processed/pune_df_enhanced.csv')

In [5]:
blr_df

,Date,DPT,AP,WS,WSD,AT,Albedo,evaporation_from_bare_soil_sum,evaporation_from_vegetation_transpiration_sum,NDVI,precipitation,surface_net_solar_radiation_sum,surface_thermal_radiation_downwards_sum,volumetric_soil_water_layer_1,LST
0,2003-01-01,12.498648,916.589625,0.608896,327.505071,294.583644,0.164460,-0.000866,-0.000029,0.265743,0.000000,1.427781e+07,31100000.0,0.192649,30.939528
1,2003-01-02,12.825784,918.004624,3.006925,290.310937,294.396121,0.164931,-0.000744,-0.000054,0.321507,0.006589,1.279035e+07,31500000.0,0.192116,31.179113
2,2003-01-03,12.970981,918.453619,2.872020,279.921010,293.651644,0.165501,-0.000771,-0.000051,0.402250,0.000000,1.492127e+07,30700000.0,0.191724,33.045705
3,2003-01-04,12.445614,917.966755,2.678422,274.952584,294.383796,0.165904,-0.000971,-0.000048,0.370532,0.000000,1.747538e+07,29400000.0,0.191393,38.266933
4,2003-01-05,14.071074,918.579933,3.104962,275.257170,294.637442,0.165906,-0.000843,-0.000050,0.407205,0.000000,1.584263e+07,30200000.0,0.190960,33.185394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6553,2020-12-25,13.500707,916.151180,2.152634,261.457477,292.513759,0.140036,-0.002592,-0.000046,0.421393,0.000000,1.671576e+07,29200000.0,0.297770,32.118758
6554,2020-12-26,12.901577,917.194971,2.312996,253.686322,292.035179,0.140023,-0.002544,-0.000056,0.421393,0.000000,1.764899e+07,28000000.0,0.289527,32.068009
6555,2020-12-27,12.950693,916.302854,2.260313,252.095788,292.771807,0.139925,-0.002661,-0.000051,0.421393,0.000000,1.757099e+07,28200000.0,0.281540,34.539327
6556,2020-12-28,10.830827,916.237697,2.003774,254.229795,292.428852,0.139896,-0.002655,-0.000055,0.421393,0.000000,1.724660e+07,28200000.0,0.273806,31.310399


In [6]:
pune_df

,Date,DPT,AP,WS,WSD,AT,Albedo,evaporation_from_bare_soil_sum,evaporation_from_vegetation_transpiration_sum,NDVI,precipitation,surface_net_solar_radiation_sum,surface_thermal_radiation_downwards_sum,volumetric_soil_water_layer_1,LST
0,2003-01-01,7.100958,941.961323,1.933461,262.137755,291.684654,0.136310,-0.000517,-0.000160,0.457425,0.000000,1.617383e+07,26800000.0,0.148610,34.246204
1,2003-01-02,10.484796,942.675433,1.436056,300.935828,294.195680,0.136204,-0.000475,-0.000130,0.359140,0.000000,1.418103e+07,30800000.0,0.148434,35.734294
2,2003-01-03,12.606104,942.881457,1.486452,300.281597,295.828579,0.136274,-0.000444,-0.000083,0.333687,0.000000,1.168451e+07,32600000.0,0.148452,32.590960
3,2003-01-04,12.646417,943.009227,1.479805,280.080236,295.986278,0.136218,-0.000461,-0.000058,0.361981,0.000000,1.190128e+07,32100000.0,0.148349,35.435619
4,2003-01-05,13.066706,943.762527,2.159695,268.877282,296.071805,0.135858,-0.000515,-0.000077,0.361361,0.000000,1.382363e+07,31200000.0,0.148208,34.235347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6553,2020-12-25,12.096190,941.053885,2.219879,278.843058,294.637357,0.127867,-0.001178,-0.000121,0.465578,0.125304,1.532697e+07,29600000.0,0.171530,33.209944
6554,2020-12-26,13.381471,942.263093,1.319014,277.169354,295.234928,0.127843,-0.001168,-0.000078,0.465578,0.000000,1.530680e+07,29900000.0,0.168792,31.352555
6555,2020-12-27,13.819016,941.287692,0.585305,157.827979,295.475096,0.127817,-0.001195,-0.000068,0.465578,0.000000,1.524516e+07,30000000.0,0.166471,33.862652
6556,2020-12-28,13.831078,940.630939,0.346885,190.861072,295.019423,0.127371,-0.001151,-0.000072,0.465578,0.000000,1.555207e+07,29500000.0,0.164420,32.318171


In [7]:
hyd_df

,Date,DPT,AP,WS,WSD,AT,Albedo,evaporation_from_bare_soil_sum,evaporation_from_vegetation_transpiration_sum,NDVI,precipitation,surface_net_solar_radiation_sum,surface_thermal_radiation_downwards_sum,volumetric_soil_water_layer_1,LST
0,2003-01-01,9.801921,952.321702,2.702138,228.989644,294.587935,0.150061,-0.000407,-0.000179,0.230621,0.0,1.548250e+07,28900000.0,0.119923,35.253027
1,2003-01-02,12.894156,953.896739,2.974241,287.669929,295.364687,0.151337,-0.000302,-0.000127,0.197456,0.0,1.177422e+07,32300000.0,0.120030,31.660535
2,2003-01-03,16.945643,954.471306,3.015529,308.536731,294.436472,0.151539,-0.000128,-0.000058,0.019037,0.0,9.142362e+06,33300000.0,0.126387,30.085849
3,2003-01-04,17.761199,954.172469,1.865083,299.395258,293.495316,0.151312,-0.000044,-0.000026,0.079633,0.0,4.418173e+06,33600000.0,0.155643,30.569864
4,2003-01-05,14.281283,954.395644,2.287668,265.010524,296.410658,0.151222,-0.000545,-0.000079,0.194655,0.0,1.449708e+07,31500000.0,0.155939,31.878895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6553,2020-12-25,13.330685,951.670802,2.015309,292.172202,293.417897,0.132338,-0.001512,-0.000082,0.249628,0.0,1.558327e+07,28400000.0,0.144402,30.396700
6554,2020-12-26,14.027632,952.521458,1.736367,284.297801,293.992337,0.132305,-0.001512,-0.000066,0.249628,0.0,1.571960e+07,28400000.0,0.143072,32.262344
6555,2020-12-27,13.223768,951.342613,1.754041,288.307250,294.178115,0.132271,-0.001620,-0.000077,0.249628,0.0,1.594387e+07,28100000.0,0.141706,33.796627
6556,2020-12-28,12.504071,951.101661,2.012608,299.893708,294.514357,0.132201,-0.001720,-0.000089,0.249628,0.0,1.591720e+07,28400000.0,0.140171,31.748070


# Since we will be applying a RNN, we will need the Date column to be Datetime column

In [8]:
blr_df['Date'] = pd.to_datetime(blr_df['Date'])
blr_df = blr_df.sort_values('Date')

In [9]:
hyd_df['Date'] = pd.to_datetime(hyd_df['Date'])
hyd_df = hyd_df.sort_values('Date')

In [10]:
pune_df['Date'] = pd.to_datetime(pune_df['Date'])
pune_df = pune_df.sort_values('Date')

# Splitting the Input and Predictor Variables

In [11]:
features = ['DPT', 'AP', 'WS', 'WSD', 'AT', 'Albedo', 'evaporation_from_bare_soil_sum', 'evaporation_from_vegetation_transpiration_sum', 'NDVI', 'precipitation', 'surface_net_solar_radiation_sum', 'surface_thermal_radiation_downwards_sum', 'volumetric_soil_water_layer_1']
target = 'LST'

In [12]:
blr_X = blr_df[features].values
blr_y = blr_df[target].values

In [13]:
hyd_X = hyd_df[features].values
hyd_y = hyd_df[target].values

In [14]:
pune_X = pune_df[features].values
pune_y = pune_df[target].values

# Scaling the data

In [15]:
scaler = StandardScaler()

In [16]:
blr_X_scaled = scaler.fit_transform(blr_X)
hyd_X_scaled = scaler.transform(hyd_X)
pune_X_scaled = scaler.transform(pune_X)

# Creating the sequences for RNN

In [17]:
def create_sequences(X, y, seq_length=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])  # predict LST at t+1
    return np.array(X_seq), np.array(y_seq)

In [18]:
seq_len = 30

In [19]:
blr_X_seq, blr_y_seq = create_sequences(blr_X_scaled, blr_y, seq_len)
hyd_X_seq, hyd_y_seq = create_sequences(hyd_X_scaled, hyd_y, seq_len)
pune_X_seq, pune_y_seq = create_sequences(pune_X_scaled, pune_y, seq_len)

# Splitting in Training and Testing Data

In [20]:
# First split: 80% training, 20% temp (for val + test)
train_idx = int(0.8 * len(blr_X_seq))
temp_idx = int(0.9 * len(blr_X_seq))  # 10% more from remaining 20%

# Split features
blr_X_train = blr_X_seq[:train_idx]
blr_X_val   = blr_X_seq[train_idx:temp_idx]
blr_X_test  = blr_X_seq[temp_idx:]

# Split targets
blr_y_train = blr_y_seq[:train_idx]
blr_y_val   = blr_y_seq[train_idx:temp_idx]
blr_y_test  = blr_y_seq[temp_idx:]

In [21]:
# First split: 80% training, 20% temp (for val + test)
train_idx = int(0.8 * len(hyd_X_seq))
temp_idx = int(0.9 * len(hyd_X_seq))  # 10% more from remaining 20%

# Split features
hyd_X_train = hyd_X_seq[:train_idx]
hyd_X_val   = hyd_X_seq[train_idx:temp_idx]
hyd_X_test  = hyd_X_seq[temp_idx:]

# Split targets
hyd_y_train = hyd_y_seq[:train_idx]
hyd_y_val   = hyd_y_seq[train_idx:temp_idx]
hyd_y_test  = hyd_y_seq[temp_idx:]

In [22]:
# First split: 80% training, 20% temp (for val + test)
train_idx = int(0.8 * len(pune_X_seq))
temp_idx = int(0.9 * len(pune_X_seq))  # 10% more from remaining 20%

# Split features
pune_X_train = pune_X_seq[:train_idx]
pune_X_val   = pune_X_seq[train_idx:temp_idx]
pune_X_test  = pune_X_seq[temp_idx:]

# Split targets
pune_y_train = pune_y_seq[:train_idx]
pune_y_val   = pune_y_seq[train_idx:temp_idx]
pune_y_test  = pune_y_seq[temp_idx:]

# Converting it into Pytorch Tensor for Further Analysis

In [23]:
blr_X_train_tensor  = torch.tensor(blr_X_train, dtype=torch.float32).to(device)
blr_y_train_tensor  = torch.tensor(blr_y_train, dtype=torch.float32).unsqueeze(1).to(device)
blr_X_test_tensor   = torch.tensor(blr_X_test, dtype=torch.float32).to(device)
blr_y_test_tensor   = torch.tensor(blr_y_test, dtype=torch.float32).unsqueeze(1).to(device)
blr_X_val_tensor  = torch.tensor(blr_X_val, dtype=torch.float32).to(device)
blr_y_val_tensor  = torch.tensor(blr_y_val, dtype=torch.float32).unsqueeze(1).to(device)

hyd_X_train_tensor  = torch.tensor(hyd_X_train, dtype=torch.float32).to(device)
hyd_y_train_tensor  = torch.tensor(hyd_y_train, dtype=torch.float32).unsqueeze(1).to(device)
hyd_X_test_tensor   = torch.tensor(hyd_X_test, dtype=torch.float32).to(device)
hyd_y_test_tensor   = torch.tensor(hyd_y_test, dtype=torch.float32).unsqueeze(1).to(device)
hyd_X_val_tensor  = torch.tensor(hyd_X_val, dtype=torch.float32).to(device)
hyd_y_val_tensor  = torch.tensor(hyd_y_val, dtype=torch.float32).unsqueeze(1).to(device)

pune_X_train_tensor = torch.tensor(pune_X_train, dtype=torch.float32).to(device)
pune_y_train_tensor = torch.tensor(pune_y_train, dtype=torch.float32).unsqueeze(1).to(device)
pune_X_test_tensor  = torch.tensor(pune_X_test, dtype=torch.float32).to(device)
pune_y_test_tensor  = torch.tensor(pune_y_test, dtype=torch.float32).unsqueeze(1).to(device)
pune_X_val_tensor = torch.tensor(pune_X_val, dtype=torch.float32).to(device)
pune_y_val_tensor = torch.tensor(pune_y_val, dtype=torch.float32).unsqueeze(1).to(device)

# Defining the ANN Model

In [24]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers,
                          dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Use the last time step
        return out

# Training the Model with the Hyperparameter Tuning done using Optuna

In [29]:
from tqdm import tqdm

def objective(trial, X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor):
    hidden_size = trial.suggest_int("hidden_size", 32, 128)
    num_layers = trial.suggest_int("num_layers", 1, 5)
    dropout = trial.suggest_float("dropout", 0.0, 0.5) if num_layers > 1 else 0.0
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    epochs = trial.suggest_int("epochs", 50, 50)
    patience = 5  # Number of epochs to wait for improvement

    model = RNNModel(input_size=X_train_tensor.shape[2], hidden_size=hidden_size,
                     num_layers=num_layers, dropout=dropout).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # Validation check
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_tensor.to(device))
            val_loss = criterion(val_preds, y_val_tensor.to(device)).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Final evaluation on test data
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_tensor.to(device))
        test_loss = criterion(test_preds, y_test_tensor.to(device)).item()

    return test_loss

# Applying the Model for Bengaluru

In [30]:
blr_study = optuna.create_study(direction="minimize")
blr_study.optimize(
    lambda trial: objective(
        trial,
        blr_X_train_tensor, blr_y_train_tensor,
        blr_X_val_tensor, blr_y_val_tensor,
        blr_X_test_tensor, blr_y_test_tensor
    ),
    n_trials=30
)

[I 2025-04-29 19:04:20,224] A new study created in memory with name: no-name-78158052-af18-452b-922f-df9a0cf2259e
[I 2025-04-29 19:04:36,781] Trial 0 finished with value: 27.81788444519043 and parameters: {'hidden_size': 93, 'num_layers': 3, 'dropout': 0.04786947539627534, 'lr': 0.00012667147135176686, 'batch_size': 16, 'epochs': 50}. Best is trial 0 with value: 27.81788444519043.


Early stopping at epoch 17


[I 2025-04-29 19:04:46,088] Trial 1 finished with value: 7.8184733390808105 and parameters: {'hidden_size': 60, 'num_layers': 1, 'lr': 0.0021697183003426524, 'batch_size': 32, 'epochs': 50}. Best is trial 1 with value: 7.8184733390808105.


Early stopping at epoch 22


[I 2025-04-29 19:04:58,713] Trial 2 finished with value: 27.349506378173828 and parameters: {'hidden_size': 62, 'num_layers': 5, 'dropout': 0.3667573921954542, 'lr': 0.009756637280143837, 'batch_size': 16, 'epochs': 50}. Best is trial 1 with value: 7.8184733390808105.


...

Early stopping at epoch 7


[I 2025-04-29 19:08:37,333] Trial 28 finished with value: 27.82585334777832 and parameters: {'hidden_size': 85, 'num_layers': 2, 'dropout': 0.11327558657678138, 'lr': 0.001311494277366446, 'batch_size': 64, 'epochs': 50}. Best is trial 20 with value: 7.142608165740967.


Early stopping at epoch 11


[I 2025-04-29 19:08:50,266] Trial 29 finished with value: 27.324871063232422 and parameters: {'hidden_size': 93, 'num_layers': 3, 'dropout': 0.20253248031507443, 'lr': 0.00015890136954446727, 'batch_size': 64, 'epochs': 50}. Best is trial 20 with value: 7.142608165740967.


Early stopping at epoch 43


In [31]:
blr_best_trial = blr_study.best_trial

In [33]:
print(f"Best MSE for Bengaluru : {blr_best_trial.value:.4f}")
print(f"Best Parameters for Bengaluru : {blr_best_trial.params}\n")

Best MSE for Bengaluru : 7.1426
Best Parameters for Bengaluru : {'hidden_size': 84, 'num_layers': 1, 'lr': 0.0024877484780444375, 'batch_size': 64, 'epochs': 50}



# Applying for Hyderabad

In [66]:
blr_model = train_final_model(
    blr_X_train_tensor, blr_y_train_tensor,
    blr_X_val_tensor, blr_y_val_tensor,
    blr_X_test_tensor, blr_y_test_tensor,
    blr_best_params
)

Early stopping triggered at epoch 25


In [67]:
blr_model.eval()

RNNModel(
  (rnn): RNN(13, 84, batch_first=True)
  (fc): Linear(in_features=84, out_features=1, bias=True)
)

In [68]:
with torch.no_grad():
    y_pred_blr = blr_model(blr_X_test_tensor).cpu().numpy()
    y_true_blr = blr_y_test_tensor.cpu().numpy()

In [69]:
metrics_blr = evaluate_predictions(y_true_blr, y_pred_blr)
print("Bangalore Metrics:")
for k, v in metrics_blr.items():
    print(f"{k}: {v:.4f}")

Bangalore Metrics:
MSE: 7.5201
MAE: 2.1464
R²: 0.7241
NSE: 0.7241
RSR: 0.5253
PBIAS: 0.0872


In [100]:
plot_predictions_plotly(y_true_blr, y_pred_blr, title="Bangalore's Prediction vs Ground Truth (Test)")

In [106]:
hyd_model = train_final_model(
    hyd_X_train_tensor, hyd_y_train_tensor,
    hyd_X_test_tensor, hyd_y_test_tensor,
    hyd_best_params
)

In [107]:
hyd_model.eval()

RNNModel(
  (rnn): RNN(5, 125, batch_first=True)
  (fc): Linear(in_features=125, out_features=1, bias=True)
)

In [ ]:
with torch.no_grad():
    y_pred_hyd = hyd_model(hyd_X_test_tensor).cpu().numpy()
    y_true_hyd = hyd_y_test_tensor.numpy()

In [109]:
metrics_hyd = evaluate_predictions(y_true_hyd, y_pred_hyd)
print("Hyderabad Metrics:")
for k, v in metrics_hyd.items():
    print(f"{k}: {v:.4f}")

Hyderabad Metrics:
MSE: 9.1709
MAE: 2.2992
R²: 0.5975
NSE: 0.5975
RSR: 0.6344
PBIAS: -1.2168


In [110]:
plot_predictions_plotly(y_true_hyd, y_pred_hyd, title="Hyderabad's Prediction vs Ground Truth (Test)")

In [111]:
pune_model = train_final_model(
    pune_X_train_tensor, pune_y_train_tensor,
    pune_X_test_tensor, pune_y_test_tensor,
    pune_best_params
)

In [112]:
pune_model.eval()

RNNModel(
  (rnn): RNN(5, 51, batch_first=True)
  (fc): Linear(in_features=51, out_features=1, bias=True)
)

In [ ]:
with torch.no_grad():
    y_pred_pune = pune_model(pune_X_test_tensor).cpu().numpy()
    y_true_pune = pune_y_test_tensor.numpy()

In [114]:
metrics_pune = evaluate_predictions(y_true_pune, y_pred_pune)
print("Pune's Metrics:")
for k, v in metrics_pune.items():
    print(f"{k}: {v:.4f}")

Pune's Metrics:
MSE: 10.3293
MAE: 2.3591
R²: 0.7715
NSE: 0.7715
RSR: 0.4780
PBIAS: -1.5183


In [115]:
plot_predictions_plotly(y_true_pune, y_pred_pune, title="Pune's Prediction vs Ground Truth (Test)")

# Storing the model for future use

In [120]:
os.makedirs('../Models/RNN', exist_ok=True)

In [ ]:
torch.save(blr_model.state_dict(), "../Models/RNN/blr_rnn2_model.pth")

In [ ]:
torch.save(hyd_model.state_dict(), "../Models/RNN/hyd_rnn2_model.pth")

In [ ]:
torch.save(pune_model.state_dict(), "../Models/RNN/pune_rnn2_model.pth")